# OpenMOSS MOSS-Image (5B DiT) Cloud LoRA Training Pipeline

一套专为 **OpenMOSS MOSS-Image (5B Single-Stream DiT)** 打造的 Kaggle 云端 LoRA 自动化极速炼丹环境。

### ⚡ 运行前必备设置 (请点击上方菜单栏中的 Settings)：
1. **Accelerator**: 必须选择 **GPU T4 x1** (16GB VRAM)；
2. **Internet**: 必须切换为 **Internet on** (用于极速拉取底模与依赖)；
3. **Data/Input**: 点击顶部 File -> Add Data 或右侧面板挂载您上传的训练图集。

In [ ]:
# 1. 验证 GPU 硬件算力与 PyTorch 原生驱动
import torch, torchvision
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("❌ 警告：未检测到 GPU！请在上方菜单栏 Settings -> Accelerator 中开启 GPU T4 x1。")

### 🎯 用户自定义超参数设置区
在此处根据您的数据集类型（画风/人物）修改对应的参数配置与触发词：

In [ ]:
# ==============================================================================
# ⚙️ 超参数控制面板 (修改下方变量即可完成个性化定制)
# ==============================================================================
LORA_NAME = "my_character"          # LoRA 产出英文名称 (如 makima, genshin_style)
TRIGGER_WORD = "my_character_char"  # 触发词 (例如: makima_char；留空 "" 则使用原标注)

REPEATS = 6                         # 单图重复次数 (人物推荐 4~6，画风推荐 8~12)
EPOCHS = 4                          # 训练轮数 (人物推荐 3~5，画风推荐 6~10)
STEPS = 400                         # 核心步数 (公式: 图片数 * REPEATS * EPOCHS // 4)

RANK = 32                           # LoRA 矩阵秩 (推荐 32)
ALPHA = 16.0                        # 缩放系数 (人物推荐 16.0，画风推荐 32.0)
LR = "1e-4"                         # 基础学习率 (Base/Heal 推荐 1e-4，SFT 画风推荐 1.2e-4~1.5e-4)

# --- 进阶云端硬件与架构控制 (已全面同步本地版最新能力) ---
MODEL_VARIANT = "heal"              # 底模分支: "heal"(推荐，数值治愈防溢出) | "base"(官方原版) | "sft"(官方SFT排版对齐版)
VRAM_PRESET = "auto"                # 显存优化档位: "auto"(智能探测) | "6g"(极限4.4G) | "8g"(均衡5.5G) | "12g"(流式8.5G) | "16g"(满血)
RESOLUTION = 1024                   # 训练目标分辨率: 1024(标准高清) | 768(速度提升约44%) | 512(快速验证)
BASE_QUANT = "auto"                 # 底模量化: "auto"(6g/8g自动用int8，12g/16g用全精度) | "int8"(极速下载省显存) | "full"
PRECISION = "auto"                  # 计算精度: "auto"(T4自动FP16，L4/A100自动BF16) | "fp16" | "bf16"
TEXT_DEVICE = "auto"                # 文本编码器提取设备: "auto"(<=10G/6g走CPU，大显存走CUDA) | "cuda" | "cpu"

# --- 续炼与二阶段微调控制 ---
TRAIN_MODE = "fresh"                # 训练模式: "fresh"(全新训练) | "resume"(断点续炼) | "init_from"(基于已训完的LoRA继续训练)
INIT_LORA_PATH = ""                 # 指定已有 .safetensors 权重路径 (留空则在 init_from/resume 模式下自动检索挂载目录)
SAVE_EVERY = 100                    # 每 N 步自动保存一次中间 checkpoint 权重 (默认 100 步)

# 留空 "" 则全自动扫描 /kaggle/input 下挂载的图集；也可显式指定绝对路径
DATASET_PATH = ""
# ==============================================================================

### 🚀 启动全自动训练循环
点击下方单元格运行，系统将自动拉取项目、下载官方修复版底模、打入兼容补丁并开启训练：

In [ ]:
import os, subprocess, sys

# 1. 获取最新代码
repo_dir = "/kaggle/working/kaggle-moss-lora-pipeline"
if not os.path.exists(repo_dir):
    !git clone https://github.com/abysschaos/kaggle-moss-lora-pipeline.git "{repo_dir}"
else:
    !cd "{repo_dir}" && git pull

%cd "{repo_dir}"

# 2. 执行训练引擎
cmd = [
    sys.executable, "train_moss.py",
    "--lora_name", str(LORA_NAME),
    "--trigger_word", str(TRIGGER_WORD),
    "--repeats", str(REPEATS),
    "--epochs", str(EPOCHS),
    "--steps", str(STEPS),
    "--rank", str(RANK),
    "--alpha", str(ALPHA),
    "--lr", str(LR),
    "--variant", str(MODEL_VARIANT),
    "--vram_preset", str(VRAM_PRESET),
    "--resolution", str(RESOLUTION),
    "--base_quant", str(BASE_QUANT),
    "--precision", str(PRECISION),
    "--text_device", str(TEXT_DEVICE),
    "--train_mode", str(TRAIN_MODE),
    "--save_every", str(SAVE_EVERY)
]
if INIT_LORA_PATH.strip():
    cmd.extend(["--init_from", INIT_LORA_PATH.strip(), "--resume_from", INIT_LORA_PATH.strip()])
if DATASET_PATH.strip():
    cmd.extend(["--dataset_dir", DATASET_PATH.strip()])

print(f"Executing: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

### 📦 成果查看与一键打包下载
训练完成后，运行下方单元格查看生成的文件，并直接在右侧面板的 **Output** 区域点击下载：

In [ ]:
import glob, os
print("\n训练产出的 LoRA 权重清单：")
for f in glob.glob(f"/kaggle/working/output/{LORA_NAME}/*"):
    size_mb = os.path.getsize(f) / (1024**2)
    print(f"  -> {os.path.basename(f)} ({size_mb:.2f} MB)")

archive_file = f"/kaggle/working/{LORA_NAME}_lora_results.zip"
if os.path.exists(archive_file):
    print(f"\n✅ 压缩包已准备就绪: {archive_file} ({os.path.getsize(archive_file)/(1024**2):.2f} MB)")
    print("请在右侧面板的 'Output' 栏目中，找到该 ZIP 文件点击下载即可！")